Jerry Zhou (zz977)
---
Kaggle Competition (Word + Character TF-IDF + Linear SVM)

**1. Import necessary libraries**

In [214]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from scipy.special import expit
from scipy.sparse import hstack

**2. Set the hyperparameters**

In [215]:
seed = 18
test_split = 0.2

n_gram_range = (2, 4)
max_features = 50000

char_analyzer = "char"
char_range = (2, 5)
char_max_features = 100000

min_freq = 3
max_freq = 1.0
char_min_freq = 5

word_lowercase = False
char_lowercase = False

word_weight = 1.25

reg_values = [0.001, 0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0]

**3. Load and split the data**

In [216]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

X = train["text"].fillna("")
y = train["label"]
test_text = test["text"].fillna("")

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_split, random_state=seed, stratify=y)

**4. Convert the text to word and character level TF-IDF features**

In [ ]:
def build_vectorizers():
    word_vec = TfidfVectorizer(analyzer="word", 
                               ngram_range=n_gram_range, 
                               max_features=max_features, 
                               min_df=min_freq, 
                               max_df=max_freq, 
                               sublinear_tf=True, 
                               lowercase=word_lowercase,
                               token_pattern=r"(?u)\b\w+\b|[^\w\s]")
    
    char_vec = TfidfVectorizer(analyzer=char_analyzer, 
                               ngram_range=char_range, 
                               max_features=char_max_features, 
                               min_df=char_min_freq, 
                               sublinear_tf=True, 
                               lowercase=char_lowercase)
    
    return word_vec, char_vec

word_vec, char_vec = build_vectorizers()

Xw_train = word_vec.fit_transform(X_train)
Xc_train = char_vec.fit_transform(X_train)
Xw_val = word_vec.transform(X_val)
Xc_val = char_vec.transform(X_val)

X_train_tfidf = hstack([word_weight * Xw_train, Xc_train]).tocsr()
X_val_tfidf = hstack([word_weight * Xw_val, Xc_val]).tocsr()

**5. Train the SVM**

In [ ]:
best_auc = 0
best_reg = None

for c in reg_values:
    model = LinearSVC(C=c, class_weight="balanced", loss="hinge", max_iter=10000)
    model.fit(X_train_tfidf, y_train)

    train_auc = roc_auc_score(y_train, model.decision_function(X_train_tfidf))
    val_auc = roc_auc_score(y_val, model.decision_function(X_val_tfidf))
    print(f"C={c}: Train AUROC {train_auc:.4f} | Validation AUROC {val_auc:.6f}")

    if best_auc < val_auc:
        best_auc, best_reg = val_auc, c

print("Best C:", best_reg, "with Validation AUROC", round(best_auc, 6))

**6. Train final model** 

In [ ]:
final_word_vec, final_char_vec = build_vectorizers()

X_full_tfidf = hstack([word_weight * final_word_vec.fit_transform(X), final_char_vec.fit_transform(X)]).tocsr()
X_test_tfidf = hstack([word_weight * final_word_vec.transform(test_text),final_char_vec.transform(test_text)]).tocsr()

final_model = LinearSVC(C=best_reg, class_weight="balanced", loss="hinge", max_iter=10000)
final_model.fit(X_full_tfidf, y)

**7. Make predictions and save results**

In [ ]:
test_scores = final_model.decision_function(X_test_tfidf)

test_predictions = expit(test_scores)

submission = pd.DataFrame({
    "id": test["id"],
    "label": np.round(test_predictions, 2)
})

submission.to_csv("submission.csv", index=False)
print(submission.shape)
print(submission.head())